In [0]:
### Import required libraries
import logging 
import traceback
import os
import sys
from datetime import datetime as dtim
import pandas as pd
import numpy as np
import re
import os.path
import math
import ast
from datetime import date,timedelta
from itertools import groupby
from pyspark.sql import *
from pyspark.sql.types import *
import pyspark.sql.functions as F 
from pyspark.sql.functions import *

In [0]:
#### Getting the execution Start Time for the DQ Tool run.

current_datetime = dtim.now()
print("current time:-", current_datetime)
current_datetime_str = str(current_datetime)
execution_time_azure_sql_str = current_datetime_str[0:19]
execution_time_delta = execution_time_azure_sql_str.replace('-','').replace(':','')
print(execution_time_delta)
print(execution_time_azure_sql_str)

current time:- 2026-01-22 13:18:19.451497
20260122 131819
2026-01-22 13:18:19


In [0]:
# creating empty dataframe for stroing error log information
log_df  = pd.DataFrame(columns=['Datetime', 'Error', "Description","Info","Dataset/column","RuleID","Dimension"])

def logs(log_df,e,info,dataset_n,ruleid,dimension):
  """
    Method helps to add the error information to dataframe for further analysis of code or execution failure.
        parameter:
            log_df: It is log_df dataframe in which we need to append new error information
            e: It is error information or we can say it is single line information about error
            info: It specify in which block of code error ocured
            dataset_n: it is dataset name and column name of i'th rule if error occured file executiong rule.
            rule_id: It is i'th rule id
            dimension: It is a dimention of i'th rule 
        return:
            Updated dataframe which has detailed information about code failure like time of failure, error, description of error, rule id if required,etc. 
  """
  
  summary = traceback.format_exc()
  now=dtim.now()
  d_time=now.strftime("%d-%m-%Y %H:%M:%S")
  exc_type, exc_value, exc_tb = sys.exc_info()
  tb = traceback.TracebackException(exc_type, exc_value, exc_tb)
  error=''.join(tb.format_exception_only())
  log = pd.DataFrame({"Datetime": [d_time], "Error": [error], "Description": [summary], "Info": [info], 'Dataset/column': [dataset_n], 'RuleID': [ruleid], 'Dimension': [dimension]})
  
  log_df = pd.concat([log_df, log], ignore_index=True)
  return log_df

In [0]:
def readData():
  """
  Method helps to read the source data, rule file, ISO file and Filter condition (if any) from ADLS container.
    return:
        All business view as seprate dataframe, rule file, iso file and filter condition file is present.
  """
 
  bvPath = "abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/"
  bvPathCRAW = "abfss://consumableraw@zukien1prdaladlsg204.dfs.core.windows.net/data/"
  bvPathClaimMI = "abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/"
  businessViewDict = {
  "Contract" : bvPath + "AdGo/contract/"
  ,"Schedule" : bvPath + "AdGo/schedule/"
  ,"Party" : bvPath + "AdGo/party/"
  ,"Party Org" : bvPath + "AdGo/partyorganisation/"
  ,"Contract_int": bvPathCRAW + "adgo/data/contract/InternalPII/"
  ,"Schedule_int": bvPathCRAW + "adgo/data/schedule/InternalPII/"
  ,"Party_int": bvPathCRAW + "adgo/data/party/InternalPII/"
  ,"policy" :bvPath + "AdGo/Policy/policy/"
  ,"policyendorsement": bvPath + "AdGo/Policy/policyendorsement/"
  ,"policy_mta": bvPath + "AdGo/Policy/policy_mta/"
  ,"partyorganisationinsured" : bvPath + "AdGo/partyorganisationinsured/"
  ,"contractpartydetails": bvPath + "AdGo/contractpartydetails/"
  ,"policypartydetails": bvPath + "AdGo/Policy/policypartydetails/"
  ,"average_rate_per_vehicle" : bvPath + "AdGo/average_rate_per_vehicle"
  ,"profit_loss_ratio_up_to_date" : bvPath + "AdGo/loss_ratio_reporting/profit_loss_ratio_up_to_date"
  ,"profit_loss_ratio_ten_mon" : bvPath + "AdGo/loss_ratio_reporting/profit_loss_ratio_ten_mon"
  }

  spark_df = {}
  for df,path in businessViewDict.items():
    spark_df[df]=spark.read.format("delta").load(path)

  # MasterRuleDF=spark.read.option("header",True).csv("adl://zukien1prdaladls05.azuredatalakestore.net/commercial_opmi/DataQualityPOC/InputFiles_New/DQTMasterRules070823.csv")
  MasterRuleDF=spark.read.option("header",True).csv("abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/Execution_Files/Source/Adgo/MasterRules_Adgo100625.csv")

  for i in MasterRuleDF.columns:
    MasterRuleDF = MasterRuleDF.withColumn(i,trim(i))

  # ISO=spark.read.option("header",True).csv("adl://zukien1prdaladls05.azuredatalakestore.net/commercial_opmi/DataQualityPOC/InputFiles_New/ISO3166.csv")
  ISO = spark.read.option("header",False).csv('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/Execution_Files/ISO3166.csv')\
  .withColumnRenamed('_c0','ISO 3166')
  ISO=list(ISO.toPandas()['ISO 3166'].str.lower())
  try:
    # FilterRulesDF = spark.read.option('header',True).csv("adl://zukien1prdaladls05.azuredatalakestore.net/commercial_opmi/DataQualityPOC/InputFiles_New/I90FilterDataCondition070823.csv")
    FilterRulesDF = spark.read.option('header',True).csv("abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/Execution_Files/Source/Adgo/FilterDataCondition_Adgo100625.csv")
    return spark_df,MasterRuleDF,FilterRulesDF,ISO
  except:
    return spark_df,MasterRuleDF,ISO

In [0]:

def cleanRules(MasterRuleDF):
  # print('clean_data cleaned Rules......')
  numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64','int','float']#never used anywhere
  index=[]
  rulelist=MasterRuleDF.toPandas()  
  rulelist.fillna(np.nan,inplace = True)
  rulelist['Rule Id'] = rulelist['Rule Id'].astype(int)
  rule_col=rulelist.columns
  for i in range(len(rulelist.dtypes)) :
      if rulelist.dtypes[i]=='float64':
          rulelist[rule_col[i]]=rulelist[rule_col[i]].astype(str)
      else:
          pass
  return rulelist

def clean_Data(Data):    
  # print('In clean_data ......')
  numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64','int','float']#never used anywhere
  index=[]
  # for part in entired_data.keys():
  data = Data.toPandas()
  # data = pd.DataFrame(entired_data)
  columns1=data.columns
  data.fillna(np.nan,inplace=True)
  if 'EXPRYP' in columns1:
    for col in columns1:
      data[col]=data[col].astype('str').str.lower()
      data[col]=data[col].astype('str').str.strip()
  else:
    for col in columns1:
      if data[col].dtype=='O':
        data[col]=data[col].astype('str').str.lower()
        data[col]=data[col].astype('str').str.strip()
        data[col]=data[col].astype('str').str.replace('-','')   # new
        #data[col]=data[col].astype('str').str.replace(',','')   # new (comma is used in underwritter display name)
        data[col]=data[col].astype('str').str.replace('&','')   # new
  
  data[:] = np.where(data == 'nan', np.nan, data)
  
    # entired_data[part] = data
  # print('clean_data entired data......')
  return data

In [0]:
def filterData(entire_data,FilterRulesDF):
    
    if FilterRulesDF.count()>=0:
        for i in FilterRulesDF.collect():
          # condition = f"{i['Variable']} {i['Condition']} {i['Criterion'].replace('[','').replace(']',''bin)}"
          condition = f"{i['Variable']} {i['Condition']} {i['Criterion']}".replace('[','').replace(']','')
          entire_data[i['Origin']] = entire_data[i['Origin']].filter(condition)

    else:
           print('No filtering required ..... ') 
    return entire_data

In [0]:
def isNan(var):
    """
     Method check if a value is equal to 'nan' or not if it is nan returns True else False
        parameter:
             var: It is data of variable/column required for i'th Rule 
        return:
            True if Variable is nan else False.
    """
    return str(var)=='nan'


def ruleDesc(dep_var_list,cond_list,criterion_list):
    """
    Method creates a rule description
      Parameters:
          dep_var_list: dependent column/variable value
          cond_list: list of i'th condition 
          criterion_list: list of i'th criterion

      return:
          It return rule description in string format.
    """
    if cond_list != "":
        rule_desc=str(dep_var_list)+" "+str(cond_list)+" "+str(criterion_list)
    else:
        rule_desc=""
    return rule_desc

In [0]:
def handleDate(current_dep_var,criterion,check,value,date_condition,condition):
    """    
    Method helps to handle date which is not in proper format and also apply given condition like add or substract days from date.
      Parameter:
          current_dep_var: data of dependent variable in dependent dataset / j'th current_dep_var from current_dep_var_list
          criterion: it is variable data from criterion 
          check: length of Criterion after spliting
          value: number of days we need to add or substract from date 
          date_condition: If condition +/- found then date_condition is +,- else 0
          condition: condition from condition column
      return:
        current_dep_var: Date column in numeric format
        condition: updated date in numeric format after applying date condition
    """
    try:
        if condition != "Range":
            check_raw=criterion.split("/")
            check_raw0=""
        else:
            check_raw0=criterion[0].split("/")
            check_raw=""
    except:
        check_raw=''
        check_raw0=''

    current_dep_var_samp=list(current_dep_var.dropna())
    check1=str(current_dep_var_samp[0]).split("/")
    check2=str(current_dep_var_samp[0]).split("-")
 

    if len(check1) == 3 or len(check2)==3:
        current_dep_var = pd.to_numeric(pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%Y%m%d'))

    else:
        pass

    if check==2 and (len(check1) == 3 or len(check2)==3) and value==0 and date_condition==0 :
        criterion=pd.to_datetime(criterion,errors='coerce').dt.strftime('%Y%m%d')
        criterion=pd.to_numeric(criterion)
    
    elif check==2 and (len(check1) == 3 or len(check2)==3) and value!=0 and date_condition!=0 :
        if date_condition=="+":
            criterion = pd.to_datetime(criterion,errors='coerce')+timedelta(days=1) 
            criterion = pd.to_numeric(criterion.dt.strftime('%Y%m%d').replace("NaT",""))
            
        elif date_condition=="-":
            criterion = pd.to_datetime(criterion,errors='coerce')-timedelta(days=1) 
            criterion = pd.to_numeric(criterion.dt.strftime('%Y%m%d').replace("NaT",""))


    if len(check_raw)==3  :
        criterion=float(criterion.replace("/",""))
    elif len(check_raw0)==3:
        criterion=[float(criterion[0].replace("/","")),float(criterion[1].replace("/",""))]
  
    return current_dep_var,criterion

In [0]:
def createEmptyErrorDF():
    """
    Method returns score and record count equal to 0 and empty dataframe.       
      return:
        score: 0
        rec_cnt:0 
        error_records: it is empty dataFrame.     
    """
    score = 0
    rec_cnt=0
    error_records = pd.DataFrame(columns=["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'] )
    return score,rec_cnt,error_records


def dimension(rulelist,i,filter_var,var,dep_var_list,cond_list,criterion_list,count):
    """
    Method is able to find score and count of filter data based on assigned dimentions and return required data in the form of list
      parameters:
          Rules : It is DataFrame which contains given Rules
          i: Rule numbers or we can say index // (len(Rules))
          filter_var : filter_var is filtered dataframe after applying all conditions
          var:it is data of feature at index i from perticular origin file at index i, from entire_data
          dep_var_list: list of all dependent variable for i'th Rule
          cond_list: list of all Condition for i'th Rule
          criterion_list: list of all Criterion for i'th Rule
      return:
          rule_score :It is list of values required for score file regenration.
          error_records : It is a DataFrame which contain records which is not satisfied conditions and criterions for Rule
    """
    typedata = rulelist['Type of Data'][i] #Data Type
    varx = rulelist['Variable'][i] # Variable to be Used
    rid = rulelist['Rule Id'][i] # Rule Id
    wt = rulelist['Weight'][i] # Weight of the Rule
    dim = rulelist['Dimension'][i] #getting the dimension of Rule
    src = rulelist['Source'][i] #getting the Source of Rule
    dd = rulelist['Data_Domain'][i] #getting the data domain of Rule

    cri = rulelist['IsCritical'][i] #gettin the IsCritical flag of rule


    rdesc = ruleDesc(dep_var_list,cond_list,criterion_list)#Rule Description
    if not isNan(rulelist["REF_COL"][i]):     
        Ref_col=rulelist["REF_COL"][i]
    else:
        Ref_col=rulelist["Variable"][i]
    op=None #Initializing Output Data for charts
    rule_score_record=None #Initializing Output Summary data
    error_records = pd.DataFrame(columns = ["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'])
    if not isNan(dim):
        if len(var)==0 :
            # for missing values, score is zero
            rule_score_record = [typedata, rulelist["REF_COL"][i],varx, rid, wt, dim, 0,0,cri,src,dd]
            error_records = pd.DataFrame(columns = ["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'])
        else:
            if dim=="Completeness":
     
                if len(filter_var) > 0 :
                    score = filter_var.count()/len(filter_var)
                    rec_cnt = filter_var.count()
                    xnew = filter_var[filter_var.isnull()]
                    error_records = pd.DataFrame(xnew)
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Validity":
                if len(filter_var) > 0:   
                    if 'null' in criterion_list:
                      score=len(filter_var)/ var.count()
                      rec_cnt = len(filter_var)
                    else:
                      score = filter_var.count()/var.count()
                      rec_cnt = filter_var.count()
                    error_records = pd.DataFrame(var[pd.Index.difference(var.index, filter_var.index)])
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Uniqueness":
                if len(filter_var) > 0 :
                    score = filter_var.nunique()/filter_var.count()
                    rec_cnt = filter_var.nunique()
                    error_records = pd.DataFrame(filter_var[filter_var.duplicated(keep=False)])
                    
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Timeliness":

                if len(var) > 0:
                    score = filter_var.count()*1.0/var.count()
                    rec_cnt = filter_var.count()
                    error_records = pd.DataFrame(var[pd.Index.difference(var.index, filter_var.index)])
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Consistency":
                if len(var) > 0:
                    score = filter_var.count()*1.0/var.count()
                    rec_cnt = filter_var.count()
                    error_records = pd.DataFrame(var[pd.Index.difference(var.index, filter_var.index)])
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Integrity":
                if len(var) > 0:
                    score = filter_var.nunique()*1.0/var.nunique()
                    rec_cnt = filter_var.nunique()
                    error_records = pd.DataFrame(var[~var.isin(filter_var)])
                else:
                    score,rec_cnt,error_records=createEmptyErrorDF()
            elif dim=="Reasonability":
                if len(var) > 0:
                    score=filter_var.count()/var.count()
                    rec_cnt=filter_var.count()
                    error_records = pd.DataFrame(var[pd.Index.difference(var.index,filter_var.index)])
                else:   
                    score,rec_cnt,error_records=createEmptyErrorDF()
            try:
              error_records.columns = ['Value']
              error_records['origIndex'] = error_records.index
              error_records['Type of Data'] = typedata
              error_records['Variable'] = varx
              error_records['Dimension'] = dim
              error_records['RuleDesc'] = rdesc
              error_records['RuleID'] = rid
              error_records['Ref Column'] = Ref_col
              error_records['Source'] = src
              error_records['Data_Domain'] = dd
              error_records['IsCritical'] = cri
            except:
              pass
            rule_score_record = [typedata, varx,Ref_col, rid, wt, dim, score, rec_cnt, cri, src,dd]
    return rule_score_record, error_records
    

In [0]:
def handleList(rulelist,i):   
    """
    Method help to convert list which is in string format into proper required list format. 
        Parameter:
            Rules: Rules as Pandas DataFrame
            i: Index of Rule 
        return:
            cond_list : list of Condition for i'th Rule
            criterion_list : list of Criterion for i'th Rule
            dep_var_list : list of dependent variable for i'th Rule
            dep_table_list :list of dependent table for i'th Rule
    """
    cond_list = ast.literal_eval(rulelist['Condition'][i])
    criterion_list = ast.literal_eval(rulelist['Criterion'][i])
    try:
      split_flag = ast.literal_eval(rulelist['Flag'][i])
    except:
      split_flag = [0 for split_l in range(len(cond_list))]

    try:
        dep_var_list=ast.literal_eval(rulelist['DepVar'][i])
    except:
        dep_var_list = [rulelist["Variable"][i] for li in range(len(cond_list))]
    # try:
    #     dep_var_list=ast.literal_eval(rulelist['DepVar'][i])
    # except:
    #     dep_var_list = [rulelist["Variable"][i] for li in range(len(cond_list))]
    # dep_var_list.append(rulelist['Variable'][i])
    # dep_var_list = list(set(dep_var_list))

    try:
        dep_table_list = ast.literal_eval(rulelist['DepTable'][i])
    except:
        dep_table_list = [rulelist["Origin"][i] for li in range(len(cond_list))]
    # dep_table_list.append(rulelist['Origin'][i])
    # dep_table_list = list(set(dep_table_list))

    return cond_list,criterion_list,dep_var_list,dep_table_list,split_flag


def createReqVarDF(var,rulelist,entire_data,dep_var_list,dep_table_list,i):
    """
    Method creates dataframe containing only the required columns. 
        paramter:
            criterion_list : it is Criterion which seprate dataframe and variable name by ':'
            entire_data: it is a dictnory where key is origin name and values is data
        return:
            temp_empty: return dataframe of variable from criterion_list
    """

    temp_col_name = var.name
    Tempo=pd.DataFrame() 
    Tempo[temp_col_name]=entire_data[rulelist["Origin"][i]][rulelist["Variable"][i]]
    # same colname issue
    for j in range(len(dep_table_list)):     
        Tempo[dep_var_list[j]]=entire_data[dep_table_list[j]][dep_var_list[j]]
    return Tempo


In [0]:
def handleColon(criterion_list,cond_list,j,entire_data):
    """
    Method helps to generate required values for creating dataset for the dependent tables mentioned in the rule file under Criterion column.
      parameters:
          cond_list : list of Condition for i'th Rule
          criterion_list : list of Criterion for i'th Rule
          j: it is index of criterion_list/cond_list element
          entire_data: it is a dictnory where key is Source name and values is data
      return:
          criterion: it is variable data from criterion 
          condition: condition from condition column
          check: length of Criterion after spliting
          value: number of days we need to add or substract from date 
          date_condition: If condition +/- found then date_condition is +,- else 0
    """
    try:
        split = criterion_list[j].split(":")
    
        condition = cond_list[j]      
        if len(split)>1 :   
            
            check = len(split)   
            split_cond1=split[1].split("+")
            split_cond2=split[1].split("-")
            if len(split_cond1)==2 :
               
                criterion=entire_data[split[0]][split_cond1[0]]
                date_condition = "+"
                value=int(split_cond1[1])
            elif len(split_cond2)==2:
                criterion=entire_data[split[0]][split_cond2[0]]
                date_condition = "-"
                value=int(split_cond2[1])
            else:
                
                criterion = entire_data[split[0]][split[1]]
              
                check=len(split)
                date_condition=0
                value=0 
        else:
            criterion = criterion_list[j]
            check=0
            date_condition=0
            value=0
    except:
        
        condition = cond_list[j]
        criterion = criterion_list[j]
        check=0
        date_condition=0
        value=0

    return condition,criterion,check,date_condition,value


In [0]:
def handleToday(criterion,value,date_condition,current_dep_var):
    """
    Method is used to convert "today" keyword into todays date and then into number format
        parameters:
            criterion: it is variable data from criterion 
            value: number of days we need to add or substract from date 
            date_condition: If condition +/- found then date_condition is +,- else 0
            current_dep_var: data of dependent variable in dependent dataset / j'th current_dep_var from current_dep_varr_list
        return:
            return all parameters after updating "today" to number format and applying conditions
    """
    current_dep_var=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%Y%m%d')
    current_dep_var=pd.to_numeric(current_dep_var)
    try:
        if criterion[0:5]=="today" and len(criterion)>5:
            date_condition=criterion[5]
            value=criterion[6:]
            if date_condition == "+":
                criterion=float(str(date.today()+timedelta(days=int(value))).replace("-",''))
            elif date_condition == "-":
                criterion=float(str(date.today()-timedelta(days=int(value))).replace("-",''))
        elif criterion[0:5]=="today" :
            criterion=float(str(date.today()).replace("-",''))
    except:
        pass
    
    return criterion,value,date_condition,current_dep_var

In [0]:
def getDateSeprator(current_dep_var):
    """
    Method is used to identify date separator based on the date format.
        parameters:
            current_dep_var: date variable data in the form of series
        return:
            return date seperator '/' or '-'
    """
    current_dep_var_samp=list(current_dep_var.dropna())
    check1=str(current_dep_var_samp[0]).split("/")
    check2=str(current_dep_var_samp[0]).split("-")
    if len(check1)==3:
        return "/"
    elif len(check2)==3:
        return "-"

  
def handleNullCriterion(criterion,current_dep_var):
    """
     Method is used to fill nan with 'Null' in current_dep_var variable
          Parameters:
              criterion : it is variable data from criterion  
              current_dep_var: data of dependent variable in dependent dataset / j'th variable in current_dep_var_list
          return:
              current_dep_var : j'th variable data after filling nan with 'Null'
    """
    try:
        if criterion=="Null":
            current_dep_var=current_dep_var.fillna("Null") # inplace=True, thsi might need to be taken 
        else:
            pass
    except:
        pass
    return current_dep_var
  

In [0]:
def filterDF(rulelist,i,check, criterion,Tempo,condition,current_dep_var,dep_var_list,j,entire_data,dep_table_list,split_flag):
    """
    Method filters tha data based on rule condition and criterion 
        parameters:
            check: length of Criterion after spliting
            criterion: it is variable data from criterion 
            required_var_df: dataframe which contain dependent columns for rule
            condition: condition from condition column
            current_dep_var: data of dependent variable in dependent dataset / j'th variable in DepVar_list
            dep_var_list: list of all dependent variable for i'th Rule
            j: index of DepVar_list element
            entire_data: it is a dictnory where key is origin name and values is data
            dep_table_list: list of all dependent tables for i'th Rule
        return:
            required_var_df: updated dataframe which contain dependent columns for rule
            filter_var: Data after filteringc on rule condition and criterion 
    """
    contract_mismatch_name=["declinedReasoncode"]
    if condition=="in":  
        if split_flag[j]==0:
          criterion=criterion
        else:
          try:
            if rulelist['Origin'][i].lower() == "contract" or rulelist['Origin'][i].lower() == "schedule":
              df = spark.read.format("delta").load("abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/AdGo_Ref/{}/{}".format(rulelist['Origin'][i].lower(),rulelist['Variable'][i].lower()))
            else:
              df = spark.read.format("delta").load("abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/AdGo_Ref/{}/{}".format("party",rulelist['Variable'][i].lower()))
          except:
            if rulelist['Origin'][i].lower() == "contract":
              for valu in contract_mismatch_name:
                if valu.lower()== rulelist['Variable'][i].lower():
                  correct_name= valu                  
                  break
            df = spark.read.format("delta").load("abfss://bv-commercial@zukien1prdaladlsg204.dfs.core.windows.net/data/AdGo_Ref/{}/{}".format(rulelist['Origin'][i].lower(),correct_name))
          temp_data=df.toPandas()
          temp_party_df=list(temp_data['Description'])
          temp_contract_df=list(temp_data['Code'])
         
          for temp_d in range(len(temp_contract_df)):
            temp_contract_df[temp_d]=temp_contract_df[temp_d].lower()
            temp_contract_df[temp_d]=temp_contract_df[temp_d].strip()
            temp_party_df[temp_d]=temp_party_df[temp_d].lower()
            temp_party_df[temp_d]=temp_party_df[temp_d].strip()
            criterion = temp_contract_df
        if check>1:
            criterion=list(criterion)
        else:
            pass      
        if 'null' in criterion:
          current_dep_var=current_dep_var.fillna('null')
        else:
          pass
        if criterion == 'ISO':
          criterion=ISO
        if rulelist["Flag"][i]=="[1]" :
          criterion=temp_contract_df
        elif rulelist["Flag"][i]=="[2]":
          criterion=temp_party_df
        
#           if current_dep_var[current_dep_var.isin(temp_contract_df) == True].count() > current_dep_var[current_dep_var.isin(temp_party_df) == True].count():
#             criterion=temp_contract_df
#           else:
#             criterion=temp_party_df

        filter_var=Tempo[current_dep_var.isin(criterion)]

    elif condition=="Range":
        filter_var=Tempo[(current_dep_var>=criterion[0]) & (current_dep_var<criterion[1])]
    
#     elif condition=='==':
#         filter_var=Tempo[current_dep_var==criterion]
    elif condition == "Contains":
        filter_var=Tempo[current_dep_var.astype(str).str.find(criterion) != -1]
      
    elif condition[0:3] == "day":
        day = pd.to_numeric(current_dep_var.astype(str).str[6:8])
        day = pd.DataFrame(list(day))
        day.columns = ["d"]
        con = condition[4:len(condition) - 1]
        d_name = "d"
        c = str(criterion)
        eval_str = "d" + con + c
        filter_var=Tempo[day.eval(eval_str)]
    elif condition[0:3] == "LEN":
        temp_var=current_dep_var.astype(str).str.replace("nan","")
        temp_var=temp_var.astype(str).str.replace(" ","")
        temp_var=temp_var.astype(str).str.len()
        condition=condition[4:-1]
        filter_var=Tempo[eval("temp_var"+condition+str(criterion))]
    elif condition=='==':
        filter_var=Tempo[current_dep_var == criterion]
    elif condition=="Format":
        if "DD" in criterion and "MM" in criterion and "YYYY" in criterion and "T" not in criterion :
            date_format={"DD":"d","MM":"m","YYYY":"Y"}
            format_list=criterion.split(" ")
            current_dep_var=entire_data[dep_table_list[j]][dep_var_list[j]]
            sep=getDateSeprator(current_dep_var)
            if sep == "/":
                current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}/%{}/%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
            elif sep == "-":
                current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}-%{}-%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
            else:
              current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}%{}%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
             
            filter_var=Tempo[current_dep_var.astype(str)==current_dep_var_temp.astype(str)]

        elif "DD" in criterion and "MM" in criterion and "YYYY" in criterion and ("T" in criterion) :
            date_format={"DD":"d","MM":"m","YYYY":"Y"}
            format_list=criterion.split(" ")
            current_dep_var=entire_data[dep_table_list[j]][dep_var_list[j]]
            sep=getDateSeprator(current_dep_var)
            if sep == "/":
                current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}/%{}/%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
            elif sep == "-":
                current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}-%{}-%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
            else:
                current_dep_var_temp=pd.to_datetime(current_dep_var,errors='coerce').dt.strftime('%{}%{}%{}'.format(date_format.get(format_list[0]),date_format.get(format_list[1]),date_format.get(format_list[2])))
            current_dep_var_t=current_dep_var.astype(str).str.split(" ").str[0]
            filter_var=Tempo[(current_dep_var_t==current_dep_var_temp.astype(str)) & (current_dep_var.astype(str).str.split(" ").str.len()==2)]
        elif criterion == "Number":
            current_dep_var=current_dep_var.astype(str).str.replace("nan","")
            current_dep_var=current_dep_var.astype(str).str.replace(".","")
            current_dep_var=current_dep_var.astype(str).str.replace(" ","")
            current_dep_var=current_dep_var.astype(str).str.replace("+","")
            filter_var=Tempo[current_dep_var.astype(str).str.isnumeric()]
        elif criterion == "Alphabet":
            current_dep_var=current_dep_var.astype(str).str.replace("nan","")
            current_dep_var=current_dep_var.astype(str).str.replace(" ","")
            filter_var=Tempo[current_dep_var.astype(str).str.isalpha()]
        elif criterion == "Email":
            current_dep_var=current_dep_var.astype(str).str.replace("nan","")
            at=current_dep_var.astype(str).str.find("@")
            dot=current_dep_var.astype(str).str.find(".")
            filter_var=Tempo[(at != -1) & (dot != -1)]
        else:
            filter_var=Tempo[current_dep_var.astype(str).str.contains(re.compile(criterion))]
            
    elif condition[0:4] == "year":
        year = pd.to_numeric(current_dep_var.astype(str).str[0:4].replace('nan',""))
        year = pd.DataFrame(list(year))
        year.columns = ["y"]
        con = condition[5:len(condition) - 1]
        y_name = "y"
        c = str(criterion)
        eval_str = "y" + con + c
        filter_var = Tempo[year.eval(eval_str)]
    elif condition[0:5]=="month":
      
        month = pd.to_numeric(current_dep_var.astype(str).str[4:6])
        month = pd.DataFrame(list(month))
        month.columns = ["m"]
        con = condition[6:len(condition) - 1]
        m_name = "m"
        c = str(criterion)
        eval_str = "m" + con + c
        filter_var = Tempo[month.eval(eval_str)]
    elif condition in [">",">=","<","<=","==","!="]:
       
        if check>0:
            if len(criterion) > len(current_dep_var):  
                current_dep_var=current_dep_var.reindex(criterion.index, fill_value=False)
            elif len(current_dep_var)>len(criterion):
                criterion=criterion.reindex(current_dep_var.index, fill_value=False)
        try:
            filter_var=Tempo[eval("current_dep_var"+condition+"criterion")]  
        except:
            filter_var=Tempo[eval("current_dep_var"+condition+criterion)]
        
            
    else:
        filter_var=pd.Series()   

    return Tempo,filter_var

In [0]:
def processvar(rulelist,i,entire_data):
    """
    This method is responsible for:
            - collecting required data from entire_data
            - cleaning data by applying function like handle_list,handle_today, handle_colon, etc
            - filtering data based on criterions and condtion by calling filterDF function
            - after getting filter data, it calls demention function to generate Data Quality Tool score and collect error files
            - finally it returns score file and error file 
        parameter:
            Rules: It is DataFrame which contains given Rules
            i: Rule numbers or we can say index // (len(Rules))
            entire_data: it is a dictnory where key is origin name and values is data
        return:
            score output and error records dataframe
    """

    var=entire_data[rulelist["Origin"][i]][rulelist["Variable"][i]]

    condition=rulelist['Condition'][i]
    count=0
    if not isNan(condition):
        cond_list,criterion_list,dep_var_list,dep_table_list,split_flag=handleList(rulelist,i)
        Tempo=createReqVarDF(var,rulelist,entire_data,dep_var_list,dep_table_list,i)

        for j in range(len(dep_table_list)):
            if count==0:
                current_dep_var = entire_data[dep_table_list[j]][dep_var_list[j]] 
                count=count+1
            else:
                current_dep_var = entire_data[dep_table_list[j]][dep_var_list[j]]
                Tempo=filter_var
                count=count+1
            
            try:
                if ":" in criterion_list[j]:
                    condition,criterion,check,date_condition,value=handleColon(criterion_list,cond_list,j,entire_data)
                 
                else:
                    criterion=criterion_list[j]
                    condition=cond_list[j]
                    value=date_condition=0
                    check=0
            except:
                if len(criterion_list)==1:
                  criterion=criterion_list[0]
                  condition=cond_list[0]
                  value=date_condition=0
                  check=0

                else:
                  criterion=criterion_list[j]
                  condition=cond_list[j]
                  value=date_condition=0
                  check=0
                
            try :    
                if criterion[0:5] =="today":
                    criterion,value,date_condition,current_dep_var=handleToday(criterion,value,date_condition,current_dep_var)
            except:
                pass
            try:
                if criterion=="Null":
                    current_dep_var=handleNullCriterion(criterion,current_dep_var)
            except:
                pass
            try:
                if condition[0:5]!="today":
                    current_dep_var,criterion=handleDate(current_dep_var,criterion,check,value,date_condition,condition)
                else:
                    pass
            except:
                pass
            Tempo,filter_var=filterDF(rulelist,i,check, criterion,Tempo,condition,current_dep_var,dep_var_list,j,entire_data,dep_table_list,split_flag)     
        filter_var = filter_var[rulelist["Variable"][i]]
        

    else:
        filter_var=var.copy()
        dep_var_list=cond_list=criterion_list=""

    rule_score_record, error_records=dimension(rulelist,i,filter_var,var,dep_var_list,cond_list,criterion_list,count)
    return rule_score_record, error_records

In [0]:
def ruleDescription(Rules,outDF_copy):
  """
  Method creates a rule description for Score File using the conditions provided in Rules File.
  parameter:
    Rules: Data From Rules File.
    outDF_Copy: It is copy of output score file in which we need to add Rule Description column. 
  retunr:
    It return outDF_copy dataframe with new column called Rule Description.
  """
  ruledesc=[]
  for i in list(outDF_copy["Rule Id"]):
      rule_desc = "Rule Id:" + str(list(Rules[Rules["Rule Id"] == i]["Rule Id"])[0]) + " | " + "Variable:" + str(
          list(Rules[Rules["Rule Id"] == i]["Variable"])[0]) + " | " + "Dimension:" + str(
          list(Rules[Rules["Rule Id"] == i]["Dimension"])[0]) + " | " + "DepTable:" + str(
          list(Rules[Rules["Rule Id"] == i]["DepTable"])[0]) + " | " + "DepVar:" + str(
          list(Rules[Rules["Rule Id"] == i]["DepVar"])[0]) + " | " + "Condition:" + str(
          list(Rules[Rules["Rule Id"] == i]["Condition"])[0]) + " | " + "Criterion:" + str(
          list(Rules[Rules["Rule Id"] == i]["Criterion"])[0])
      ruledesc.append(rule_desc.replace("nan", " "))
  outDF_copy["Rule Description"]=ruledesc
  return outDF_copy

In [0]:
def EDA_datasetup(contract_df,party_df,party_org_df,schedule_df,commercial_queue_df,commercial_agent_df,i90policydetailed_df,i90policyunique_df,i90policytransaction_df,actpolicydetailed_df,actactivepolicy_df,actmastertransaction_df,actmasterunique_df):
  contract_df["LastModifiedDate"]=contract_df["LastModifiedDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["LastUserModifiedDate"]=contract_df["LastUserModifiedDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["SubmissionCreatedDate"]=contract_df["SubmissionCreatedDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["SubmissionReceiptDate"]=contract_df["SubmissionReceiptDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["BrokerDeadLineDate"]=contract_df["BrokerDeadLineDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["QuoteCreatedOn"]=contract_df["QuoteCreatedOn"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["QuoteIssuedDate"]=contract_df["QuoteIssuedDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["QuoteExpiryDate"]=contract_df["QuoteExpiryDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["InceptionDate"]=contract_df["InceptionDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["ExpiryDate"]=contract_df["ExpiryDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["ContractClosedDate"]=contract_df["ContractClosedDate"].astype("datetime64[ns]",errors= 'ignore')
  contract_df["ClaimExperienceDate"]=contract_df["ClaimExperienceDate"].astype("datetime64[ns]",errors= 'ignore')

  party_org_df["LastModifiedDate"]=party_org_df["LastModifiedDate"].astype("datetime64[ns]",errors= 'ignore')
  party_org_df["DateLastChecked"]=party_org_df["DateLastChecked"].astype("datetime64[ns]",errors= 'ignore')

  party_df["LastModifiedDate"]=party_df["LastModifiedDate"].astype("datetime64[ns]",errors= 'ignore')
  party_df["LastModifiedDate"]=party_df["LastModifiedDate"].astype("datetime64[ns]",errors= 'ignore')
  party_df["CreatedDate"]=party_df["CreatedDate"].astype("datetime64[ns]",errors= 'ignore')
  party_df["DateLastChecked"]=party_df["DateLastChecked"].astype("datetime64[ns]",errors= 'ignore')

  schedule_df["InceptionDate"]=schedule_df["InceptionDate"].astype("datetime64[ns]",errors= 'ignore')
  schedule_df["LastModifiedDate"]=schedule_df["LastModifiedDate"].astype("datetime64[ns]",errors= 'ignore')

  commercial_queue_df["consumableraw_loaddate"]=commercial_queue_df["consumableraw_loaddate"].astype("datetime64[ns]",errors= 'ignore')
  commercial_queue_df["Date"]=commercial_queue_df["Date"].astype("datetime64[ns]",errors= 'ignore')

  commercial_agent_df["Day"]=commercial_agent_df["Day"].astype("datetime64[ns]",errors= 'ignore')
  commercial_agent_df["Logged_on_Time"]=commercial_agent_df["Logged_on_Time"].astype("datetime64[ns]",errors= 'ignore')
  commercial_agent_df["Logged_off_Time"]=commercial_agent_df["Logged_off_Time"].astype("datetime64[ns]",errors= 'ignore')

  i90policydetailed_df['INCPTP']=i90policydetailed_df['INCPTP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['EXPRYP']=i90policydetailed_df['EXPRYP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['OINCPP']=i90policydetailed_df['OINCPP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['CANDTP']=i90policydetailed_df['CANDTP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['STDATP']=i90policydetailed_df['STDATP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['DTEFFP']=i90policydetailed_df['DTEFFP'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['POLDTR']=i90policydetailed_df['POLDTR'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['IDDATR']=i90policydetailed_df['IDDATR'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['RTERMR']=i90policydetailed_df['RTERMR'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['PTERMR']=i90policydetailed_df['PTERMR'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['source_event_date']=i90policydetailed_df['source_event_date'].astype("datetime64[ns]",errors= 'ignore')
  i90policydetailed_df['Renewal_Date']=i90policydetailed_df['Renewal_Date'].astype(str).astype(float).astype(int).astype("datetime64[ns]",errors= 'ignore')

  i90policyunique_df['POLDTR']=i90policyunique_df['POLDTR'].astype("datetime64[ns]",errors= 'ignore')
  i90policyunique_df['IDDATR']=i90policyunique_df['IDDATR'].astype("datetime64[ns]",errors= 'ignore')
  i90policyunique_df['CANDTP']=i90policyunique_df['CANDTP'].astype("datetime64[ns]",errors= 'ignore')
  i90policyunique_df['INCPTP']=i90policyunique_df['INCPTP'].astype("datetime64[ns]",errors= 'ignore')
  i90policyunique_df['EXPRYP']=i90policyunique_df['EXPRYP'].astype("datetime64[ns]",errors= 'ignore')
  i90policyunique_df['Renewal_Date']=i90policyunique_df['Renewal_Date'].astype(str).astype(float).astype(int).astype("datetime64[ns]",errors= 'ignore')

  i90policytransaction_df['POLDTR']=i90policytransaction_df['POLDTR'].astype("datetime64[ns]",errors= 'ignore')
  i90policytransaction_df['IDDATR']=i90policytransaction_df['IDDATR'].astype("datetime64[ns]",errors= 'ignore')

  actpolicydetailed_df['raw_event_date']=actpolicydetailed_df['raw_event_date'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['consumableraw_loaddate']=actpolicydetailed_df['consumableraw_loaddate'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['inception_date_key']=actpolicydetailed_df['inception_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['accepted_date_key']=actpolicydetailed_df['accepted_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['renewal_date_key']=actpolicydetailed_df['renewal_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['effective_date_key']=actpolicydetailed_df['effective_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['term_end_date_key']=actpolicydetailed_df['term_end_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_ps']=actpolicydetailed_df['raw_event_date_ps'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_be']=actpolicydetailed_df['raw_event_date_be'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_agt']=actpolicydetailed_df['raw_event_date_agt'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_prd']=actpolicydetailed_df['raw_event_date_prd'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_canc']=actpolicydetailed_df['raw_event_date_canc'].astype("datetime64[ns]",errors= 'ignore')
  actpolicydetailed_df['raw_event_date_pyt']=actpolicydetailed_df['raw_event_date_pyt'].astype("datetime64[ns]",errors= 'ignore')

  actactivepolicy_df['inception_date_key']=actactivepolicy_df['inception_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actactivepolicy_df['accepted_date_key']=actactivepolicy_df['accepted_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actactivepolicy_df['renewal_date_key']=actactivepolicy_df['renewal_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actactivepolicy_df['effective_date_key']=actactivepolicy_df['effective_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actactivepolicy_df['term_end_date_key']=actactivepolicy_df['term_end_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actactivepolicy_df['policy_band_effective_nbrn']=actactivepolicy_df['policy_band_effective_nbrn'].astype("datetime64[ns]",errors= 'ignore')

  actmastertransaction_df['inception_date_key']=actmastertransaction_df['inception_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['accepted_date_key']=actmastertransaction_df['accepted_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['renewal_date_key']=actmastertransaction_df['renewal_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['effective_date_key']=actmastertransaction_df['effective_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['term_end_date_key']=actmastertransaction_df['term_end_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['cancellation_date']=actmastertransaction_df['cancellation_date'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['policy_band_effective_mta']=actmastertransaction_df['policy_band_effective_mta'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['policy_band_effective_cn']=actmastertransaction_df['policy_band_effective_cn'].astype("datetime64[ns]",errors= 'ignore')
  actmastertransaction_df['policy_band_effective_nbrn']=actmastertransaction_df['policy_band_effective_nbrn'].astype("datetime64[ns]",errors= 'ignore')



  actmasterunique_df['inception_date_key']=actmasterunique_df['inception_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmasterunique_df['accepted_date_key']=actmasterunique_df['accepted_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmasterunique_df['renewal_date_key']=actmasterunique_df['renewal_date_key'].astype("datetime64[ns]",errors= 'ignore')
  actmasterunique_df['effective_date_key']=actmasterunique_df['effective_date_key'].astype("datetime64[ns]",errors= 'ignore')

  return contract_df,party_df,party_org_df,schedule_df,commercial_queue_df,commercial_agent_df,i90policydetailed_df,i90policyunique_df,i90policytransaction_df,actpolicydetailed_df,actactivepolicy_df,actmastertransaction_df,actmasterunique_df

In [0]:
class EDA:                             # EDA for Input data
    def datatype(self, data, thres):     # this function is to find datatype for a column

        datatype = []
        unique_cols=["ContractId","ContractReference","CaseReference","I90PolicyNumber","PrimdaryPhone","ScheduleId","UPRN","OccupationCode","Longitude","Latitude","policy_agent_key","policy_key"]
        for col in data.columns:
            dtype = data[col].dtypes
            null = data[col].isnull().sum()
            size = data[col].shape[0]
            try:
              unique = data[col].unique()
              d_c=list(data[col].unique())
            except:
              data[col]=data[col].astype(str)
              unique = data[col].unique()
        
              d_c=list(data[col].unique())

            if size != null:
                if (dtype == "<M8[ns]" or dtype == "datetime64[ns]"):
                    datatype.append("datetime")
                    
                elif (dtype == object or dtype == "O"):     # if dtype is object or 0
                    if len(unique) <= thres:
                        datatype.append("cat")
                    else:
                        datatype.append("Unique")

                elif dtype == "int64" or dtype == "float64" or dtype == int or dtype == float or dtype == "int32" or dtype=="float32":  # if dtype is int or float
                    check = 0
                    try:
                        for j in range(data[col].shape[0]):
                            if int(data[col][j]) == data[col][j]:     # checking whether values are continious (just checking count of float values)
                                pass
                            else:
                                check = check + 1
                                break
                    except:                                            # To handle errors occured by null values
                        pass
                    if len(unique)<thres:
                        datatype.append("cat")
                    elif len(unique_cols) == 0:
                        if check != 0:
                          datatype.append("numeric")         # if unique count greater than threshold and have float values then it's numeric
                        else:                        # To classify Unique data type (ID)  or numeric                    
                          datatype.append("Unique")      # if no null values then for Unique  data type size of column should be equal to count of unique
                    else:
                        if col in unique_cols:
                          datatype.append("Unique")
                        else:
                          
                          datatype.append("numeric")

                elif (dtype != "int64" and dtype != "float64" and dtype != "O"and dtype != "<M8[ns]" and dtype!= "int32" and dtype != "float32")  :    # if dtype is none of the mentioned types then considering as datetime
                        datatype.append(None)                        # else data is not recognized/have less than 3 values  so considering as None

                else:
                       # not recognized columns are considering as None
                    datatype.append(None)

            else:
                datatype.append(None)       # if whole column is null dtype is considering as None
            
        return datatype

    def percentile(self, data_col, arr_p):                       # arr_p is list percentiles ( example : 5 (5th percentile))
        perc = [np.percentile(data_col, p) for p in arr_p]       # finding arr_p percentile values  of data
        return perc

    def bins(self, data): 
        dist = data.dropna().sort_values()
        q1 = dist.quantile(0.25)
        q3 = dist.quantile(0.75)                               # finding no of bins need to considered
        iqr = q3 - q1
        bin_width = (2 * iqr) / (len(dist) ** (1 / 3))
        # bin_count = int(np.ceil((np.percentile(dist,99) - np.percentile(dist,1)) / bin_width))
        if bin_width > 0:
          bin_count = int(np.ceil((np.percentile(dist,99) - np.percentile(dist,1)) / bin_width))
        else:
          dist = data[data.astype(int)!=0].sort_values()
          q1 = dist.quantile(0.25)
          q3 = dist.quantile(0.75)                               # finding no of bins need to considered
          iqr = q3 - q1
          bin_width = (2 * iqr) / (len(dist) ** (1 / 3))
          bin_count = int(np.ceil((np.percentile(dist,99) - np.percentile(dist,1)) / bin_width))
        return dist, bin_count 

    def split(self, data_var, n_bin):
        if n_bin <=25:
            pass
        else:
            n_bin=25
        maxx=np.percentile(data_var,99)
        minn=np.percentile(data_var,1)
        r=maxx-minn
        l = [(minn) + (r / n_bin) * binn for binn in range(n_bin+1)]   # splitting as per the bin length
        l= [data_var.min()] +l+ [data_var.max()]                                                
        m = [[l[binn], l[binn + 1]] for binn in range(len(l) - 1)]     # appending bin range (min,max)
        
        bin1 = []
        for binn in list(data_var):
            for binnj in range(len(l)):
                if l[binnj] > binn:                                    # assigning unique value for each bin to find freq (example : if value lies 5th bin then append 5)
                    bin1.append(binnj)
                    break
        bin1.sort() 
#         sort assigned values
        k = [len(list(group)) for key, group in groupby(bin1)]
        uniq = np.unique(bin1)
      
        bin2 = []
        count = 0
      
        for app in range(1, len(m)+1):
            if app in uniq:                                          # finding frequency of bins
                bin2.append(k[count])
                count = count + 1
            else:  
               # There is a possibility of no value exist in a particular bin so appending 0 to it
                bin2.append(0)
     
        final = self.rounding(m)        
       # calling rounding function to round values
        change = bin2.pop()
        bin2.append(change + 1)                                      # + 1 is adding to last bin because max value exist
        return final, bin2

    def rounding(self, m):
        final = []
        for i in m:
            for j in range(2, 200):                                 # rounding values lie between 2 to 10
                if np.round(i[1], j) - np.round(i[0], j) == 0:           # round value as per the float values (iter until  maximum value of bin - minimum value of bin ==  0 )      
                  if i[1]==i[0]:
                    final.append([np.round(i[0], j), np.round(i[1], j)])
                    break
                  else:
                    pass
                else:
                    final.append([np.round(i[0], j), np.round(i[1], j)])  # round it if it fails condition
                    break
        return final

    def EDA_Final(self, data, dtype, sheet):
        #Initializing data frames
        Data_Frame_cat = pd.DataFrame(
            columns=['Dataset', 'Variable', 'Value', 'Condition', 'Value Count'])
        Data_Frame_num = pd.DataFrame(
            columns=['Dataset', 'Variable', 'condition', 'result',  'Reference'])
        Data_Frame_date = pd.DataFrame(
            columns=['Dataset', 'Variable', 'Year', 'Month', 'Year Month', 'Monthly Count', 'Minimum Value',
                     'Maximum Value', 'Unique Day Count'])
        Data_Frame_Unique = pd.DataFrame(
            columns=['Dataset', 'Variable', 'Unique_Count' , 'Null_Count', 'Total_count'])
        for i in range(len(dtype)):
            if dtype[i] == "cat":                                 # If dtype is categorical
              
#                 unique = data[data.columns[i]].unique().tolist()  # unique values
                unique_val=data[data.columns[i]].value_counts().index.tolist()
                unique=[]

                for un in range(len(unique_val)):
                  unique.append(unique_val[un])
                 
                null = data[data.columns[i]].isnull().sum()       # null count
                val_c = data[data.columns[i]].value_counts().tolist() # value counts
                freq=[null]+val_c
                unique=[np.nan]+unique
                sheet_name = []
                variable = []
                condition = []
                Size = []
                col_T = []
                for k in range(len(freq)):
                    sheet_name.append(sheet)                   # appending sheet name
                    variable.append(data.columns[i])           # column name
                    condition.append("count")                  # appending condition
                sample_df_cat = pd.DataFrame([sheet_name, variable, unique, condition, freq]).T
                sample_df_cat.columns = ['Dataset', 'Variable', 'Value', 'Condition', 'Value Count']
                Data_Frame_cat = pd.concat([Data_Frame_cat, sample_df_cat]) # concatinating column EDA data  to initialized  data frame
            elif dtype[i] == "Unique":
                 # concatinating column EDA data  to initialized  
                count=data[data.columns[i]].count()
                null=data[data.columns[i]].isna().sum()
                unique=data[data.columns[i]].nunique()
                sheet_name= sheet
                variable = data.columns[i]
                sample_df_Unique = pd.DataFrame([sheet_name, variable, unique, null, count]).T
                sample_df_Unique.columns = ['Dataset', 'Variable', 'Unique_Count' , 'Null_Count', 'Total_count']
                Data_Frame_Unique = pd.concat([Data_Frame_Unique, sample_df_Unique])
            elif dtype[i] == "numeric":                  # if dtype is numeric
                sheet_name = []
                variable = []
                Total = []
                null = []
                deciles = []
                freq = []
                print(sheet,data.columns[i])
                minn = np.min(list(data[data.columns[i]].dropna()))   # minimum value
                maxx = np.max(list(data[data.columns[i]].dropna()))   # Maximum value
                mean = data[data.columns[i]].mean(skipna=True) # mean
                std = data[data.columns[i]].std(skipna=True) # standard deviation
                arr_p = [1, 5, 10, 25, 50, 75, 90, 95, 99]       # percentile values
                temp_d=data[data.columns[i]].dropna()
                perc = self.percentile(temp_d, arr_p) # calling percentile function to find percentiles
                condition = ["Minimum", "Maximum", "Arithmetic Mean", "Standard Deviation"]        # condition
                data_var, n_bins = self.bins(data[data.columns[i]])   # calling bin function to find number of bins
                r = maxx - minn   #range
                deciles, freq = self.split(data_var, n_bins)  # calling split function for deciles and their frequencies
                null_val = [data[data.columns[i]].isnull().sum()]       # null count
                arr_p_string = ["1st Percentile", "5th Percentile", "10th Percentile", "1st Quartile", "Median",
                                "3rd Quartile", "90th Percentile", "95th Percentile", "99th Percentile"]
                condition = condition + deciles + ["Null"] + arr_p_string
                col_T = []
                for c in range(len(condition)):
                    sheet_name.append(sheet)              # sheet name
                    variable.append(data.columns[i])      #column name
                result = [minn, maxx, mean, std]
             
                result = result + freq + null_val + perc     # results for each condition

                ref = []
                ref_dict = {"1st Percentile": "01", "5th Percentile": "02", "10th Percentile": "03",
                            "1st Quartile": "04", "Median": "05", "3rd Quartile": "06", "90th Percentile": "07",
                            "95th Percentile": "08", "99th Percentile": "09", "Minimum": "10", "Maximum": "11",
                            "Arithmetic Mean": "12", "Standard Deviation": "13"}
                for refer in condition:

                    try:
                        ref.append(ref_dict.get(refer, None))   # Reference column for powerbi
                    except:
                        ref.append(None)

                sample_df_num = pd.DataFrame([sheet_name, variable, condition, result, ref]).T
                sample_df_num.columns = ['Dataset', 'Variable', 'condition', 'result',
                                         "Reference"]
                Data_Frame_num = pd.concat([Data_Frame_num, sample_df_num]) # concatinating column EDA data  to initialized numeric data frame


            elif dtype[i] == "datetime":        # if dtype is datetime
                year = []
                month = []
                minimum = []
                maximum = []
                count = []
                unique_c = []
                month_year = []
                data[data.columns[i]] = pd.to_datetime(data[data.columns[i]],errors='coerce').dt.strftime('%d/%m/%Y')  # making data in standard date format
                data[data.columns[i]] = pd.to_datetime(data[data.columns[i]], errors='coerce')         # errors to ignore errors caused by null values
                y = list(pd.DatetimeIndex(data[data.columns[i]]).year)  # list of years
                m = list(pd.DatetimeIndex(data[data.columns[i]]).month) # list of months
                d = list(pd.DatetimeIndex(data[data.columns[i]]).day)   # list of days
                try:
                    for dt in range(data[data.columns[i]].shape[0]):

                        data[data.columns[i]] = data[data.columns[i]].astype("O")
                        max_i = list(pd.DataFrame(
                            {"Date": data[data.columns[i]], "Year": y, "Month": m, "day": d}).dropna().groupby(
                            ["Year", "Month"]).agg({"day": ["max", "min", "count"]}).apply(list).to_dict()[
                                         ('day', 'max')].items())                             # maximum value
                        mini_i = list(pd.DataFrame(
                            {"Date": data[data.columns[i]], "Year": y, "Month": m, "day": d}).dropna().groupby(
                            ["Year", "Month"]).agg({"day": ["max", "min", "count"]}).apply(list).to_dict()[
                                          ('day', 'min')].items())                            # minimum value
                        v_cou_i = list(pd.DataFrame(
                            {"Date": data[data.columns[i]], "Year": y, "Month": m, "day": d}).dropna().groupby(
                            ["Year", "Month"]).agg({"day": ["max", "min", "count"]}).apply(list).to_dict()[
                                           ('day', 'count')].items())                        # count of values in paricular year-month
                        unique = list(pd.DataFrame(
                            {"Date": data[data.columns[i]], "Year": y, "Month": m, "day": d}).dropna().groupby(
                            ["Year", "Month"]).agg({"day": ["nunique"]}).apply(list).to_dict()[
                                          ('day', 'nunique')].items())                     # number of unique values in a month
                        year.append(max_i[dt][0][0])                                       # appending year  for everu unique date
                        month.append(max_i[dt][0][1])                                      # month for everu unique date
                        if len(str(int(month[-1]))) == 1:
                            month_year.append(str(int(year[-1])) + " " + "0" + str(int(month[-1])))
                        else:                                                                             # month year combined for powerbi axis
                            month_year.append(str(int(year[-1])) + " " + str(int(month[-1])))

                        minimum.append(str(int(mini_i[dt][1])) + "-" + str(int(month[-1])) + "-" + str(int(year[-1])))    # minimum date in each month
                        maximum.append(str(int(max_i[dt][1])) + "-" + str(int(month[-1])) + "-" + str(int(year[-1])))     # maximum date in each month
                        count.append(v_cou_i[dt][1])          #count
                        unique_c.append(unique[dt][1])        # unique count

                except:
                    pass

                null = data[data.columns[i]].isnull().sum()
                year.append(None)
                month.append(None)
                minimum.append(None)             # appending values for null row
                maximum.append(None)
                count.append(null)
                unique_c.append(0)

                sheet_name = []
                variable = []
                total = []
                col_T = []
                for m in range(len(year)):
                    sheet_name.append(sheet)   # sheet name
                    variable.append(data.columns[i]) # column name
                sample_df_date = pd.DataFrame(
                    [sheet_name, variable,  year, month, month_year, count, minimum, maximum, unique_c]).T
                sample_df_date.columns = ['Dataset', 'Variable', 'Year', 'Month', 'Year Month',
                                          'Monthly Count', 'Minimum Value', 'Maximum Value', 'Unique Day Count']

                Data_Frame_date = pd.concat([Data_Frame_date, sample_df_date])     # concatinating column EDA data  to initialized date data frame  
        return Data_Frame_cat, Data_Frame_num, Data_Frame_date,Data_Frame_Unique

    def global_df(self, shtlst, entire_data):       # Function for global data analysis
        data_shape = []
        data_column = []
        for sheet in shtlst:
            data = entire_data[sheet]
            data_shape.append(data.shape[0])       # size of the data
            data_column.append(len(data.columns))  # number of columns
        sheets = shtlst                            # sheet name
        global_df = pd.DataFrame([sheets, data_shape, data_column]).T
        global_df.columns = ["Dataset", "Total Records", "Total Columns"]
        return global_df

    def main(self, shtlst, entire_data, thres):   # main function (where all functions are called)
        catego = pd.DataFrame()
        numer = pd.DataFrame()
        datetim = pd.DataFrame()
        Unique=pd.DataFrame()
        for sheet in shtlst:                     #  for data in each sheet
            data = entire_data[sheet]
            dtype = self.datatype(data, thres)   # find datatype
            a, b, c, d = self.EDA_Final(data, dtype, sheet)  # EDA for categorical ,numeric,datetime
            global_DF = self.global_df(shtlst, entire_data) # Global data details
            catego = pd.concat([catego, a])
            numer = pd.concat([numer, b])
            datetim = pd.concat([datetim, c])
            Unique= pd.concat([Unique,d])
        return catego, numer, datetim, global_DF , Unique      # return all 4 data frames (category,numeric,datetime,global)


In [0]:
import datetime

In [0]:
def find_completeness(simple_com_rule,sparkalldf,outDF):
  com_dim_score = pd.DataFrame()
  for i in simple_com_rule['Origin'].unique():
      sim_com_rule = simple_com_rule[simple_com_rule['Origin'].isin([f"{i}"])].reset_index(drop=True)
      req_data =  sparkalldf[i]
      columns=list(sim_com_rule["Variable"])
      rec_cnt_list = req_data.select([count(when(col(c).isNotNull(),c)).alias(c) for c in columns])
      null_count_list = req_data.select([count(when(col(c).isNull(),c)).alias(c) for c in columns])
      rec_cnt=list(rec_cnt_list.collect()[0])
      null_cnt=list(null_count_list.collect()[0])
      score = [] # declaration of the list  
      for x in range (0, len(rec_cnt)):  
        score.append( rec_cnt[x]/( rec_cnt[x]+ null_cnt[x]) ) 
      # Main_df=seprule.drop(["DepTable","DepVar","Criterion","Condition"],axis=1) 
      com_score = pd.DataFrame()   
      com_score['Type of Data'] = list(sim_com_rule['Type of Data'])
      com_score['Variable'] = list(sim_com_rule['Variable'])
      com_score['Ref Column'] = list(sim_com_rule['REF_COL'])
      com_score['Rule Id'] = list(sim_com_rule['Rule Id'])
      com_score['Weight'] = list(sim_com_rule['Weight'])
      com_score['Dimension'] = list(sim_com_rule['Dimension'])
      com_score["Score"] = score
      com_score["Record Count"] = rec_cnt
      com_score['IsCritical'] = list(sim_com_rule['IsCritical'])
      com_score['Source'] = list(sim_com_rule['Source'])
      com_score['Data_Domain'] = list(sim_com_rule['Data_Domain'])
      com_dim_score = pd.concat([com_dim_score, com_score]).sort_values("Rule Id").reset_index(drop=True)  #new
      #com_dim_score = com_dim_score.append(com_score).sort_values("Rule Id").reset_index(drop=True)       #old
    # comDF = pd.concat([com_dim_score,com_dim_score]).sort_values("Rule Id").reset_index(drop=True)
  outDF = pd.concat([outDF,com_dim_score]).sort_values("Rule Id").reset_index(drop=True)
  return outDF    

In [0]:
def find_uniqueness(rules,entired_data,outDF,errorDF,log_df):    
  out = []
  for i in range(len(rules)):   
    try:
      typedata = rules['Type of Data'][i] #Data Type
      var = rules['Variable'][i] # Variable to be Used
      rid = rules['Rule Id'][i] # Rule Id
      wt = rules['Weight'][i] # Weight of the Rule
      dim = rules['Dimension'][i] #getting the dimension of Rule
      src = rules['Source'][i] #getting the Source of Rule
      dd = rules['Data_Domain'][i] #getting the data domain of Rule
      crt = rules['IsCritical'][i]
      if not isNan(rules["REF_COL"][i]):     
          Ref_col=rules["REF_COL"][i]
      else:
          Ref_col=rules["Variable"][i]

      if entired_data[rules['Origin'][i]].count()>0:
        # rec_cnt = entired_data[rules['Origin'][i]].select(col(var).isNotNull()).distinct().count()
        # score = rec_cnt/(entired_data[rules['Origin'][i]].select(col(var).isNotNull()).count())
        rec_cnt = entired_data[rules['Origin'][i]].select(var).distinct().where(col(var).isNotNull()).count()
        total_cnt = entired_data[rules['Origin'][i]].select(var).where(col(var).isNotNull()).count()
        if total_cnt >0:
          score = rec_cnt/total_cnt
        else:
          score = 0
      rule_score_record = [typedata, var,Ref_col, rid, wt, dim, score, rec_cnt,crt,src,dd]
      out.append(rule_score_record)

    except Exception as e:
        
        log_df=logs(log_df,e,"Rule Failed",rulelist['Origin'][i]+"/"+rulelist["Variable"][i],rulelist['Rule Id'][i],rulelist["Dimension"][i])
        rule_score_record=['Failed', 'Failed', 'Failed', rules['Rule Id'][i], 'Failed', 'Failed', 'Failed','Failed', 'Failed','Failed','Failed']
        out.append(rule_score_record)
        error_records = pd.DataFrame(columns = ["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'])
        if error_records is not None:
            errorDF = pd.concat([errorDF, error_records], axis=0)

  unique_dim_score = pd.DataFrame(out,columns=['Type of Data', 'Variable', 'Ref Column', 'Rule Id', 'Weight',
        'Dimension', 'Score', 'Record Count','IsCritical', 'Source', 'Data_Domain'])
  outDF = pd.concat([outDF,unique_dim_score]).sort_values("Rule Id").reset_index(drop=True)
  return outDF,errorDF,log_df

In [0]:
def req_data_to_pandas(rulelist):
  entire_data ={}
  for i in rulelist['Origin'].unique():
    req_rule = rulelist[rulelist['Origin'].isin([f"{i}"])].reset_index(drop=True)
    if i in ['Party','Contract','Party Org']:
      req_data =  sparkall_df[i]
    else:
      var_col = []
      
      if not req_rule['DepVar'].isnull:
        for j in req_rule['DepVar']:
          var_col.append(j)
      else:
        for j in req_rule['DepVar']:
          try:
           var_col.append(ast.literal_eval(j)[0])
          except:
            pass

      # if not req_rule['DepVar'].isnull:
      #   for j in req_rule['DepVar']:
      #     var_col.append(j)
      for k in req_rule['Variable']:
        var_col.append(k)
      for l in range(len(req_rule['Criterion'])):
        if ":" in req_rule['Criterion'][l]:
          if '+' in req_rule['Criterion'][l]:
            cri = ast.literal_eval(req_rule['Criterion'][l].split('+')[0]+"']")
            var_col.append(cri[0].split(':')[1])
          
          elif '-' in req_rule['Criterion'][l]:
            cri = ast.literal_eval(req_rule['Criterion'][l].split('-')[0]+"']")
            var_col.append(cri[0].split(':')[1])
          else:
            cri = ast.literal_eval(req_rule['Criterion'][l])
            var_col.append(cri[0].split(':')[1])
      var_col = list(set(var_col))
      req_data =  sparkall_df[i].select(var_col)
    req_data_df=clean_Data(req_data)
    entire_data[i]=req_data_df
  return entire_data 

In [0]:
def find_conditional_score(rulelist,entire_data,outDF,errorDF,log_df):
  error_check=0
  for i in range(len(rulelist)):
    try:
      rule_score_record, error_records = processvar(rulelist,i,entire_data)#using method to process the rule
    except Exception as e:
      error_check=error_check+1
      log_df=logs(log_df,e,"Rule Failed",rulelist['Origin'][i]+"/"+rulelist["Variable"][i],rulelist['Rule Id'][i],rulelist["Dimension"][i])
      rule_score_record=['Failed', 'Failed', 'Failed', rulelist['Rule Id'][i], 'Failed', 'Failed', 'Failed','Failed', 'Failed','Failed','Failed']
      error_records = pd.DataFrame(columns = ["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'])
    if rule_score_record is not None:
        # if there is an output for rule score, append it to Rule Score DataFrame
        outDF.loc[outDF.shape[0]] =  rule_score_record
    if error_records is not None:
        errorDF = pd.concat([errorDF, error_records], axis=0)
  return outDF,errorDF,log_df


In [0]:
def calculateScore(rulelist,sparkall_df,log_df):
  """
  Method helps to call the function per dimension
  """
  error_check=0
  outDF = pd.DataFrame(columns = ['Type of Data','Variable' , 'Ref Column','Rule Id', 'Weight', 'Dimension', 'Score','Record Count','IsCritical','Source','Data_Domain'])
  errorDF = pd.DataFrame(columns = ["RuleID",'Value', 'origIndex', 'Type of Data', 'Variable', "Ref Column",'Dimension', 'RuleDesc','IsCritical','Source','Data_Domain'])

  if len(rulelist.loc[(rulelist['Dimension'] == 'Uniqueness')])>0:
    uniqueness_rule = rulelist[rulelist['Dimension'].isin(["Uniqueness"])].reset_index(drop=True)
    rulelist = rulelist[~rulelist['Dimension'].isin(["Uniqueness"])].reset_index(drop=True)
    outDF,errorDF,log_df = find_uniqueness(uniqueness_rule,sparkall_df,outDF,errorDF,log_df)    

  if len(rulelist.loc[(rulelist['Dimension'] == 'Completeness') & (rulelist['Condition'].isnull())])>0:
    simple_com_rule = rulelist.loc[(rulelist['Dimension'] == 'Completeness') & (rulelist['Condition'].isnull())].reset_index(drop=True)
    rulelist = rulelist[~rulelist['Rule Id'].isin(simple_com_rule['Rule Id'])].reset_index(drop=True)
    outDF = find_completeness(simple_com_rule,sparkall_df,outDF)

  entire_data = req_data_to_pandas(rulelist)
  outDF,errorDF,log_df = find_conditional_score(rulelist,entire_data,outDF,errorDF,log_df)

  return outDF,errorDF,log_df

In [0]:
try:
  sparkall_DF,MasterRuleDF,FilterRulesDF,ISO=readData()
  sparkall_df = filterData(sparkall_DF,FilterRulesDF) 
except:
  sparkall_df,MasterRuleDF,ISO=readData()

rulelist=cleanRules(MasterRuleDF)
outDF,errorDF,log_df = calculateScore(rulelist,sparkall_df,log_df)
outDF_copy=outDF.copy()
rulelist=cleanRules(MasterRuleDF)
outDF_copy=ruleDescription(rulelist,outDF_copy)
outDF_copy['Score'] = outDF_copy['Score'].astype(str)
outDF_copy['Record Count'] = outDF_copy['Record Count'].astype(str)
outDF_copy['IsCritical'] = outDF_copy['IsCritical'].astype(str)
Out_sparkdf=spark.createDataFrame(outDF_copy)
Out_sparkdf=Out_sparkdf.withColumn("Score",Out_sparkdf.Score.cast(FloatType()))
Out_sparkdf=Out_sparkdf.withColumn("Record Count",Out_sparkdf['Record Count'].cast(IntegerType()))
Out_sparkdf=Out_sparkdf.withColumn("IsCritical",Out_sparkdf['IsCritical'].cast(IntegerType()))
Out_sparkdf=Out_sparkdf.withColumn("Rule Id",Out_sparkdf["Rule Id"].cast(IntegerType()))
Out_sparkdf=Out_sparkdf.withColumn("Weight",Out_sparkdf['Weight'].cast(IntegerType()))
Out_sparkdf=Out_sparkdf.withColumn("Execution_Time",F.lit(execution_time_delta))
Out_sparkdf=Out_sparkdf.withColumn("Execution_Time_Azure_SQL",F.lit(execution_time_azure_sql_str))
Out_sparkdf=Out_sparkdf.withColumn("Execution_Time_Azure_SQL",F.to_timestamp('Execution_Time_Azure_SQL','yyyy-MM-dd HH:mm:ss'))
errorDF['Value'] = errorDF['Value'].astype(str)

/home/spark-7c1d9783-da1f-4d77-b1e3-fc/.ipykernel/244219/command-1667542759599066-2543927902:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rulelist.fillna(np.nan,inplace = True)
/home/spark-7c1d9783-da1f-4d77-b1e3-fc/.ipykernel/244219/command-1667542759599066-2543927902:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if rulelist.dtypes[i]=='float64':
/home/spark-7c1d9783-da1f-4d77-b1e3-fc/.ipykernel/244219/command-1667542759599066-2543927902:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as l

In [0]:
display(Out_sparkdf)

Type of Data,Variable,Ref Column,Rule Id,Weight,Dimension,Score,Record Count,IsCritical,Source,Data_Domain,Rule Description,Execution_Time,Execution_Time_Azure_SQL
Contract,crawuniqueid,AdGo CRAW Unique Identifier,113,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:113 | Variable:crawuniqueid | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,EventType,AdGo Event Type,114,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:114 | Variable:EventType | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractId,AdGo Contract ID,116,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:116 | Variable:ContractId | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractReference,AdGo Contract Reference,117,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:117 | Variable:ContractReference | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,LastModifiedBy,AdGo Last User Modified By,127,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:127 | Variable:LastModifiedBy | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,LastUserModifiedBy,AdGo Last User Modified By,129,1,Completeness,0.99999285,139568,1,ADGO,Policy,Rule Id:129 | Variable:LastUserModifiedBy | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,InsuredPartyReference,AdGo Insured Party Reference,137,1,Completeness,0.99756396,139229,1,ADGO,Policy,Rule Id:137 | Variable:InsuredPartyReference | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ExpiryDate,AdGo Expiry Date,149,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:149 | Variable:ExpiryDate | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,crawuniqueid,AdGo CRAW Unique Identifier,167,1,Uniqueness,1.0,139569,1,ADGO,Policy,Rule Id:167 | Variable:crawuniqueid | Dimension:Uniqueness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractId,AdGo Contract ID,170,1,Uniqueness,1.0,139569,1,ADGO,Policy,Rule Id:170 | Variable:ContractId | Dimension:Uniqueness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z


In [0]:
Out_sparkdf.filter("IsCritical ==1").display()

Type of Data,Variable,Ref Column,Rule Id,Weight,Dimension,Score,Record Count,IsCritical,Source,Data_Domain,Rule Description,Execution_Time,Execution_Time_Azure_SQL
Contract,crawuniqueid,AdGo CRAW Unique Identifier,113,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:113 | Variable:crawuniqueid | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,EventType,AdGo Event Type,114,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:114 | Variable:EventType | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractId,AdGo Contract ID,116,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:116 | Variable:ContractId | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractReference,AdGo Contract Reference,117,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:117 | Variable:ContractReference | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,LastModifiedBy,AdGo Last User Modified By,127,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:127 | Variable:LastModifiedBy | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,LastUserModifiedBy,AdGo Last User Modified By,129,1,Completeness,0.99999285,139568,1,ADGO,Policy,Rule Id:129 | Variable:LastUserModifiedBy | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,InsuredPartyReference,AdGo Insured Party Reference,137,1,Completeness,0.99756396,139229,1,ADGO,Policy,Rule Id:137 | Variable:InsuredPartyReference | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ExpiryDate,AdGo Expiry Date,149,1,Completeness,1.0,139569,1,ADGO,Policy,Rule Id:149 | Variable:ExpiryDate | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,crawuniqueid,AdGo CRAW Unique Identifier,167,1,Uniqueness,1.0,139569,1,ADGO,Policy,Rule Id:167 | Variable:crawuniqueid | Dimension:Uniqueness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract,ContractId,AdGo Contract ID,170,1,Uniqueness,1.0,139569,1,ADGO,Policy,Rule Id:170 | Variable:ContractId | Dimension:Uniqueness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z


In [0]:
Out_sparkdf.filter(col("Rule Id") >= 3015).display()

Type of Data,Variable,Ref Column,Rule Id,Weight,Dimension,Score,Record Count,IsCritical,Source,Data_Domain,Rule Description,Execution_Time,Execution_Time_Azure_SQL
Contract_int,consumableraw_loaddate,AdGo Consumable Raw Loaddate,3015,1,Completeness,1.0,3075816,1,ADGO,Policy,Rule Id:3015 | Variable:consumableraw_loaddate | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Schedule_int,consumableraw_loaddate,AdGo Consumable Raw Loaddate,3016,1,Completeness,1.0,2435167,1,ADGO,Policy,Rule Id:3016 | Variable:consumableraw_loaddate | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Party_int,consumableraw_loaddate,AdGo Consumable Raw Loaddate,3017,1,Completeness,1.0,1135751,1,ADGO,Policy,Rule Id:3017 | Variable:consumableraw_loaddate | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
policy,PolicyNumber,AdGo Policy Number,3018,1,Completeness,0.9691308,2386,1,ADGO,Policy,Rule Id:3018 | Variable:PolicyNumber | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
policyendorsement,PolicyNumber,AdGo Policy Number,3019,1,Completeness,0.9712251,6818,1,ADGO,Policy,Rule Id:3019 | Variable:PolicyNumber | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
policy_mta,PolicyNumber,AdGo Policy Number,3020,1,Completeness,0.972652,2703,1,ADGO,Policy,Rule Id:3020 | Variable:PolicyNumber | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
policyendorsement,PolicyStatus,AdGo Policy Status,3021,1,Completeness,1.0,7020,1,ADGO,Policy,Rule Id:3021 | Variable:PolicyStatus | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Contract_int,raw_datalake_file,AdGo Raw Datalake File,3056,1,Completeness,1.0,3075816,1,ADGO,Policy,Rule Id:3056 | Variable:raw_datalake_file | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Schedule_int,raw_datalake_file,AdGo Raw Datalake File,3057,1,Completeness,1.0,2435167,1,ADGO,Policy,Rule Id:3057 | Variable:raw_datalake_file | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z
Party_int,raw_datalake_file,AdGo Raw Datalake File,3058,1,Completeness,1.0,1135751,1,ADGO,Policy,Rule Id:3058 | Variable:raw_datalake_file | Dimension:Completeness | DepTable: | DepVar: | Condition: | Criterion:,20260122 131819,2026-01-22T13:18:19Z


In [0]:
# Writing the score into azure as a delta table
Out_sparkdf = Out_sparkdf.select([F.col(col).alias(col.strip(' ').replace(' ', '_')) for col in Out_sparkdf.columns])

Out_sparkdf.coalesce(1).write.mode('overwrite').format('delta').option('mergeSchema','true').save('abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DQ_Score/Source/Adgo')


In [0]:
%sql
--Writing the score into hive_metastore as an External table
CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.dq_tool.adgo_dq_score USING DELTA LOCATION 'abfss://bv-commercial@zukien1prdaladlsg205.dfs.core.windows.net/data/data_quality/DQ_Score/Source/Adgo'